In [ ]:
!pip install langchain
!pip install openai
!pip install tiktoken
!pip install faiss-gpu
!pip install langchain_experimental
!pip install "langchain[docarray]"
!pip install -U langchain-openai
!pip install transformers
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.2 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.6
    Uninstalling langchain-text-splitters-0.3.6:
      Successfully uninstalled langchain-text-splitters-0.3.6
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.20
    Uninstalling langchain-0.3.20:
      Successfully uninstalled langchain-0.3.20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.2 MB/s eta 0:0

In [ ]:
import os
from langchain.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
DATA_PATH = '/cancer'
loader = DirectoryLoader(DATA_PATH, loader_cls=TextLoader)
data = loader.load()

# Split the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function = len, add_start_index = True)
data = text_splitter.split_documents(data)

# Use OpenAI Embeddings
embeddings = OpenAIEmbeddings()

# Create the vectorstore
vectorstore = FAISS.from_documents(data, embedding=embeddings)

# Set up the language model (GPT-4)
llm = ChatOpenAI(temperature=0.7, model_name="gpt-4o-mini")

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Use the "stuff" chain type for basic retrieval
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5})  # Adjust k for the number of top results
)

# Example Query
query = "I just got diagnosed with cancer, and I feel overwhelmed. What are the first steps I should take?"
result = qa_chain.run(query)

sentiment_result = list(classifier(query)[0].values())[0]
if sentiment_result in ["sadness", "fear","surprise"]:
  result += " Cancer can be a tough journey, but one's courage and determination will always shine brighter than any challenge faced. The road may be hard, but no one walks it alone. There are people who care, and there is always hope for brighter days ahead."

# Print the answer
answer = result
print(answer)


Device set to use cpu


I'm sorry to hear about your diagnosis. It's normal to feel overwhelmed. Here are some steps that may help you navigate this challenging time:

1. **Talk to Your Cancer Care Team**: Discuss your diagnosis and treatment options with your healthcare providers. They can help address your concerns and provide information tailored to your situation.

2. **Educate Yourself**: Learn as much as you can about your type of cancer, treatment plans, and potential side effects. Knowledge can help you feel more in control.

3. **Plan for Support**: Reach out to family and friends for emotional support. Consider joining support groups or talking to a counselor to help you cope with your feelings.

4. **Prepare for Treatment**: Stock your pantry with your favorite foods and consider cooking meals in advance to make things easier during treatment.

5. **Manage Your Health**: Discuss any dietary changes or nutritional supplements with your care team, especially if you anticipate side effects from treatm

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
print(list(classifier("Tell me abt lung cancer")[0].values())[0])

Device set to use cpu


neutral
